# 3 - Evaluate + benchmark: base + RAG vs RAFT on one T4

Kaggle: GPU T4 x2, Internet on. Notebook 1's output is an optional input - if it isn't mounted, this notebook rebuilds the data (same seed, same splits). The RAFT model is pulled from my Hugging Face repo.

Models, all fed by the same retriever:
- Qwen2.5-1.5B base + RAG
- my RAFT 1.5B + RAG
- Qwen2.5-3B + RAG
- Qwen2.5-7B (4-bit AWQ) + RAG

To keep it fair: same retrieval query, same top-k passages, greedy decoding. Base models get the plain prompt, RAFT gets the format it was trained on. Anything timed runs with `CUDA_VISIBLE_DEVICES=0`, so exactly one T4.

Order: install -> write files -> set paths -> (smoke test) -> 3.1-3.4 generate -> 3.5 score -> 3.6/3.7 benchmark -> 3.8 judge -> 3.9 report -> 3.10 read samples.

If vLLM OOMs while capturing CUDA graphs, add `--eager` to that command.

Where this fits: notebook 1 built the data and index, notebook 2 trained the RAFT model, this notebook decides whether it was worth it - on quality *and* on cost.

In [ ]:
!pip -q install vllm sentence-transformers pyyaml matplotlib huggingface_hub\nimport os; os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
import os
os.makedirs('/kaggle/working/proj/configs', exist_ok=True)
os.makedirs('/kaggle/working/proj/src', exist_ok=True)
%cd /kaggle/working/proj

In [ ]:
%%writefile /kaggle/working/proj/configs/config.yaml
# RAFT vs RAG on a single T4. The same config is written in all three notebooks so every stage reads the same numbers.
seed: 42

data:
  doc2dial_url: https://doc2dial.github.io/file/doc2dial_v1.0.1.zip
  holdout_doc_frac: 0.10     # docs never seen in training -> "unseen-doc" test set
  passage_words: 140         # target passage size when chunking sections
  history_turns: 4           # previous turns shown in the prompt
  max_train: 8000
  max_valid: 200
  max_test_seen: 500
  max_test_unseen: 300
  valid_dialogue_frac: 0.04

retrieval:
  embed_model: BAAI/bge-small-en-v1.5
  query_prefix: "Represent this sentence for searching relevant passages: "
  save_top: 20               # stored per query (hard negatives + recall@k)
  k: 4                       # docs given to the model at test time

raft:
  n_docs: 4                  # docs in each training prompt
  p_oracle: 0.8              # gold passage is in the context 80% of the time; the RAFT paper found 60-100% all work
  hard_neg_frac: 0.7         # share of distractors taken from retriever top-10
  style: cot                 # cot = quote then answer | direct = answer only
  max_tokens: 1900           # drop longer examples so nothing gets truncated mid-answer at max_length 2048

train:
  base_model: Qwen/Qwen2.5-1.5B-Instruct     # stretch: Qwen/Qwen2.5-3B-Instruct
  max_length: 2048
  epochs: 2
  per_device_batch: 2        # x grad_accum 8 = effective batch 16. use 1 for the 3B model
  grad_accum: 8
  lr: 2.0e-4
  warmup_ratio: 0.03
  lora_r: 32
  lora_alpha: 64             # alpha = 2r, the usual starting point
  lora_dropout: 0.05
  save_steps: 200
  eval_steps: 200

eval:
  max_model_len: 4096
  gpu_util: 0.88             # share of VRAM vLLM grabs up front (weights + KV cache)
  max_new_tokens_direct: 256
  max_new_tokens_cot: 384

bench:
  n_queries: 200
  warmup: 10
  # rough on-demand T4 price, not a quote. cost per 1k queries is linear in it, so it's easy to redo
  gpu_usd_per_hour: 0.35

In [ ]:
%%writefile /kaggle/working/proj/src/common.py
"""Shared helpers: config, IO, prompts, answer parsing, metrics, retriever.

Everything that must be identical between the RAG baseline and the RAFT model
(prompt template, retrieval query, metrics) lives here so the comparison is fair.
"""
from __future__ import annotations

import json
import random
import re
import string
from collections import Counter
from pathlib import Path


# ----------------------------------------------------------------------------- IO
# read the yaml config into a plain dict - every script starts with this
def load_cfg(path: str = "configs/config.yaml") -> dict:
    import yaml

    with open(path) as f:
        return yaml.safe_load(f)


# fix every random source we use (python, numpy, torch) so reruns give the same splits and samples
def set_seed(seed: int) -> None:
    random.seed(seed)
    try:
        import numpy as np

        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch

        torch.manual_seed(seed)
    except ImportError:
        pass


# one JSON object per line -> list of dicts (blank lines skipped)
def read_jsonl(path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


# opposite of read_jsonl; creates the folder if it doesn't exist yet
def write_jsonl(path, rows) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


# save a single object (stats, reports) as pretty-printed JSON
def write_json(path, obj) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


# ------------------------------------------------------------------------ prompts
# system prompt when documents are given. the 'say so if the docs don't have it' line matters:
# RAFT trains with missing gold passages, so the model should learn to admit it
SYSTEM = (
    "You are a customer-support agent. Use the reference documents to answer the "
    "customer's last message. Be accurate and concise, and do not invent policies "
    "or numbers. If the documents do not contain the answer, say so."
)
# closed-book version, used when there are no documents in the prompt
SYSTEM_CLOSED = "You are a customer-support agent. Answer the customer's last message accurately and concisely."
# extra instruction for the cot style: quote the evidence first, then answer
COT_INSTR = (
    "First copy the sentences you rely on between ##begin_quote## and ##end_quote##, "
    "then write the reply after ##Answer:"
)


# turn the conversation into the search query used by the retriever
def retrieval_query(history: list[dict]) -> str:
    """Dialogue-aware query: the last two customer turns."""
    # a follow-up like "how do I do that?" means nothing on its own, so the previous customer turn comes along
    users = [t["text"] for t in history if t["role"] == "user"]
    return " ".join(users[-2:])


# one passage as the model sees it: [n] title - section, then the text
def fmt_passage(i: int, p: dict) -> str:
    head = p["title"] + (f" - {p['section']}" if p.get("section") else "")
    return f"[{i}] {head}\n{p['text']}"


# build the chat prompt: documents (if any) + conversation so far + what to write.
# used for training AND evaluation, so RAFT is tested on exactly the format it learned
def build_messages(history: list[dict], passages: list[dict], style: str = "direct") -> list[dict]:
    parts = []
    if passages:
        parts.append("Reference documents:\n" + "\n\n".join(fmt_passage(i + 1, p) for i, p in enumerate(passages)))
    conv = "\n".join(f"{'Customer' if t['role'] == 'user' else 'Agent'}: {t['text']}" for t in history)
    parts.append("Conversation so far:\n" + conv)
    parts.append("Write the agent's next reply." + (" " + COT_INSTR if style == "cot" else ""))
    return [
        {"role": "system", "content": SYSTEM if passages else SYSTEM_CLOSED},
        {"role": "user", "content": "\n\n".join(parts)},
    ]


# the text the model is trained to produce: quote + answer (cot) or just the answer (direct)
def build_target(style: str, quote: str, response: str) -> str:
    # doc2dial already gives the grounding span, so the quote half of the target is free (no teacher model)
    if style == "cot":
        return f"##Reason: The relevant text says ##begin_quote## {quote} ##end_quote##.\n##Answer: {response}"
    return response


# grabs whatever sits between the quote markers
_QUOTE_RE = re.compile(r"##begin_quote##(.*?)##end_quote##", re.S)


# split a model output into the final answer and its quotes - only the answer gets scored
def parse_output(text: str) -> tuple[str, list[str]]:
    """Return (final answer, quoted spans). Works for direct and cot outputs."""
    quotes = [q.strip() for q in _QUOTE_RE.findall(text)]
    # base models never write ##Answer:, so fall back to the whole text minus the markers
    if "##Answer:" in text:
        ans = text.rsplit("##Answer:", 1)[1]
    else:
        ans = _QUOTE_RE.sub("", text).replace("##Reason:", "")
    return ans.strip(), quotes


# ------------------------------------------------------------------------ metrics
# articles and stop words are ignored when comparing texts
_ARTICLES = re.compile(r"\b(a|an|the)\b")
_STOP = set(
    "a an the and or but if of to in on for with at by from as is are was were be been it this that these those "
    "you your i we our they their he she his her not no do does did can will would should could may might have has had "
    "so than then there here what which who whom when where how about into over under up out".split()
)


# clean a string before comparing, so 'The SSA.' and 'ssa' count as the same
def normalize(s: str) -> str:
    # SQuAD-style: lowercase, drop punctuation and articles
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = _ARTICLES.sub(" ", s)
    return " ".join(s.split())


# word-overlap F1 between our reply and the human agent's reply (0 = nothing shared, 1 = same words)
def token_f1(pred: str, gold: str) -> float:
    p, g = normalize(pred).split(), normalize(gold).split()
    if not p or not g:
        return float(p == g)
    common = Counter(p) & Counter(g)
    n = sum(common.values())
    if n == 0:
        return 0.0
    prec, rec = n / len(p), n / len(g)
    return 2 * prec * rec / (prec + rec)


# like F1 but based on the longest common subsequence, so word order counts a bit
def rouge_l(pred: str, gold: str) -> float:
    # plain LCS DP; the 200-token cap keeps it cheap when a model rambles
    p, g = normalize(pred).split()[:200], normalize(gold).split()[:200]
    if not p or not g:
        return float(p == g)
    prev = [0] * (len(g) + 1)
    for a in p:
        cur = [0]
        for j, b in enumerate(g):
            cur.append(prev[j] + 1 if a == b else max(prev[j + 1], cur[j]))
        prev = cur
    lcs = prev[-1]
    if lcs == 0:
        return 0.0
    prec, rec = lcs / len(p), lcs / len(g)
    return 2 * prec * rec / (prec + rec)


# how much of the answer is backed by the documents we passed in - low value = likely made up
def groundedness(answer: str, context: str) -> float:
    """Share of content words in the answer that appear in the supplied context (hallucination proxy)."""
    # crude on purpose: word overlap, not entailment. good for comparing models side by side,
    # not as an absolute hallucination rate
    a = [w for w in normalize(answer).split() if w not in _STOP]
    if not a:
        return 1.0
    c = set(normalize(context).split())
    return sum(w in c for w in a) / len(a)


# did the model quote text that really exists in the context, or invent it?
def quote_grounded(quotes: list[str], context: str) -> float | None:
    """Share of quoted spans that really occur in the context (None if no quotes)."""
    # only the cot model quotes - this checks it isn't inventing them
    if not quotes:
        return None
    c = normalize(context)
    return sum(normalize(q) in c for q in quotes if normalize(q)) / max(1, sum(1 for q in quotes if normalize(q)))


# p50 / p95 for latency numbers (linear interpolation, same idea as numpy.percentile)
def percentile(xs: list[float], q: float) -> float:
    xs = sorted(xs)
    if not xs:
        return float("nan")
    k = (len(xs) - 1) * q / 100
    lo, hi = int(k), min(int(k) + 1, len(xs) - 1)
    return xs[lo] + (xs[hi] - xs[lo]) * (k - lo)


# ---------------------------------------------------------------------- retriever
# dense retriever: embed all passages once, then a query is one matrix multiply + top-k.
# the exact same object is used for building RAFT data, RAG eval and the latency benchmark
class Retriever:
    """Dense retriever (bge-small by default). Whole corpus is a few thousand passages -> brute force matmul."""

    # load the embedding model and embed the corpus (or load the cached embeddings)
    def __init__(self, passages: list[dict], embed_model: str, prefix: str, device: str | None = None, emb_path=None):
        import numpy as np
        import torch
        from sentence_transformers import SentenceTransformer

        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.prefix = prefix
        self.passages = passages
        self.model = SentenceTransformer(embed_model, device=self.device)
        if self.device == "cuda":
            self.model.half()
        # embeddings are cached to disk so eval / bench don't re-embed the corpus every time
        if emb_path and Path(emb_path).exists():
            emb = torch.from_numpy(np.load(emb_path))
        else:
            texts = [f"{p['title']} - {p['section']}\n{p['text']}" for p in passages]
            emb = self.model.encode(texts, batch_size=64, normalize_embeddings=True, convert_to_tensor=True, show_progress_bar=True).float().cpu()
            if emb_path:
                Path(emb_path).parent.mkdir(parents=True, exist_ok=True)
                np.save(emb_path, emb.numpy())
        self.emb = emb.to(self.device).half() if self.device == "cuda" else emb

    # top-k passage indices + similarity scores for many queries, in batches of `bs`
    def search_batch(self, queries: list[str], k: int, bs: int = 128) -> tuple[list[list[int]], list[list[float]]]:
        idxs, scores = [], []
        for i in range(0, len(queries), bs):
            q = self.model.encode([self.prefix + x for x in queries[i : i + bs]], normalize_embeddings=True, convert_to_tensor=True, show_progress_bar=False)
            sims = q.to(self.emb.dtype) @ self.emb.T
            s, ix = sims.topk(min(k, sims.shape[1]), dim=1)
            idxs += ix.cpu().tolist()
            scores += s.float().cpu().tolist()
        return idxs, scores

    # single-query version, used in the benchmark where queries arrive one by one
    def search(self, query: str, k: int) -> list[int]:
        return self.search_batch([query], k)[0][0]

In [ ]:
%%writefile /kaggle/working/proj/src/run_eval.py
"""Stage 6 - generate answers with vLLM (fp16, T4) for one model across several context modes.

Runs:  rag:K    top-K retrieved passages     closed   no documents     oracle   gold passage(s) only
    python src/run_eval.py --model Qwen/Qwen2.5-1.5B-Instruct --tag base   --style direct --splits test_seen test_unseen --runs rag:4 closed oracle rag:2 rag:8
    python src/run_eval.py --model work/merged/raft_cot        --tag raft  --style cot    --splits test_seen test_unseen --runs rag:4 closed oracle rag:2 rag:8
One process per model (vLLM does not reliably free GPU memory inside a process).
"""
from __future__ import annotations

import argparse
from pathlib import Path

from common import build_messages, load_cfg, parse_output, read_jsonl, write_jsonl


# which passages go into the prompt for a given run mode
def context_pids(ex: dict, run: str, retrieved: list[str]) -> list[str]:
    # closed = floor (model's own knowledge), oracle = ceiling (perfect retrieval), rag:k = what would actually ship
    if run == "closed":
        return []
    if run == "oracle":
        return ex["gold_pids"][:2]
    if run.startswith("rag:"):
        return retrieved[: int(run.split(":")[1])]
    raise ValueError(run)


# load one model in vLLM, then for each split x run mode: build prompts, generate, save everything
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/config.yaml")
    ap.add_argument("--data_dir", default="work/data")
    ap.add_argument("--out_dir", default="work/preds")
    ap.add_argument("--model", required=True)
    ap.add_argument("--tag", required=True, help="name used in result tables, e.g. base / raft_cot / qwen7b_awq")
    ap.add_argument("--style", default="direct", choices=["direct", "cot"])
    ap.add_argument("--splits", nargs="+", default=["test_seen", "test_unseen"])
    ap.add_argument("--runs", nargs="+", default=["rag:4"])
    ap.add_argument("--limit", type=int, default=None)
    ap.add_argument("--quantization", default=None, help="awq for the optional 7B baseline")
    ap.add_argument("--eager", action="store_true", help="enforce_eager=True: use if CUDA-graph capture OOMs")
    a = ap.parse_args()
    cfg = load_cfg(a.config)
    ev = cfg["eval"]
    dd = Path(a.data_dir)

    from vllm import LLM, SamplingParams

    llm = LLM(model=a.model, dtype="half", max_model_len=ev["max_model_len"], gpu_memory_utilization=ev["gpu_util"],
              quantization=a.quantization, enforce_eager=a.eager, seed=cfg["seed"])
    tok = llm.get_tokenizer()
    # greedy, so reruns give the same answers and the models are compared like for like
    sp = SamplingParams(temperature=0.0, max_tokens=ev["max_new_tokens_cot" if a.style == "cot" else "max_new_tokens_direct"])
    by_pid = {p["pid"]: p for p in read_jsonl(dd / "passages.jsonl")}

    for split in a.splits:
        rows = read_jsonl(dd / f"{split}.jsonl")[: a.limit]
        ret = {r["id"]: r["retrieved"] for r in read_jsonl(dd / f"ret_{split}.jsonl")}
        for run in a.runs:
            # all prompts for this split/mode go to vLLM in one batch - much faster than one at a time
            ctxs = [context_pids(ex, run, ret[ex["id"]]) for ex in rows]
            prompts = [tok.apply_chat_template(build_messages(ex["history"], [by_pid[p] for p in c], a.style), tokenize=False, add_generation_prompt=True)
                       for ex, c in zip(rows, ctxs)]
            outs = llm.generate(prompts, sp)
            # keep the raw output, parsed answer, context and token counts - metrics.py scores from these
            recs = []
            for ex, c, o in zip(rows, ctxs, outs):
                raw = o.outputs[0].text
                ans, quotes = parse_output(raw)
                recs.append({"id": ex["id"], "tag": a.tag, "split": split, "run": run, "style": a.style,
                             "context_pids": c, "gold_pids": ex["gold_pids"], "gold_in_context": bool(set(ex["gold_pids"]) & set(c)),
                             "context_text": "\n".join(by_pid[p]["text"] for p in c), "prompt_tokens": len(o.prompt_token_ids),
                             "out_tokens": len(o.outputs[0].token_ids), "raw": raw, "answer": ans, "quotes": quotes,
                             "gold_response": ex["response"], "history": ex["history"]})
            fn = Path(a.out_dir) / f"{a.tag}__{split}__{run.replace(':', '')}.jsonl"
            write_jsonl(fn, recs)
            print("wrote", fn, len(recs))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/proj/src/metrics.py
"""Stage 7 - score every predictions file. Writes results/metrics.json.

Per (tag, split, run):  token-F1 (+95% bootstrap CI), ROUGE-L, groundedness of the answer in the supplied
context, quote-grounded rate (RAFT cot only: do the quoted spans really exist in the context?),
share of queries whose gold passage was actually in the context, mean prompt / output tokens.

    python src/metrics.py --pred_dir work/preds --out results/metrics.json
"""
from __future__ import annotations

import argparse
import random
from pathlib import Path

from common import groundedness, quote_grounded, read_jsonl, rouge_l, token_f1, write_json


# 95% confidence interval for the mean, by resampling the per-query scores 1000 times
def bootstrap_ci(xs: list[float], n: int = 1000, seed: int = 0) -> tuple[float, float]:
    # resample the per-query F1s - tells me whether a 0.01 gap between two models is real or just noise
    rng = random.Random(seed)
    means = sorted(sum(rng.choices(xs, k=len(xs))) / len(xs) for _ in range(n))
    return means[int(0.025 * n)], means[int(0.975 * n)]


# all metrics for one predictions file (one model x split x run mode)
def score(rows: list[dict]) -> dict:
    f1 = [token_f1(r["answer"], r["gold_response"]) for r in rows]
    rl = [rouge_l(r["answer"], r["gold_response"]) for r in rows]
    gr = [groundedness(r["answer"], r["context_text"]) for r in rows if r["context_text"]]
    qg = [q for q in (quote_grounded(r["quotes"], r["context_text"]) for r in rows if r["context_text"]) if q is not None]
    lo, hi = bootstrap_ci(f1)
    n = len(rows)
    return {"n": n, "f1": round(sum(f1) / n, 4), "f1_lo": round(lo, 4), "f1_hi": round(hi, 4), "rougeL": round(sum(rl) / n, 4),
            "groundedness": round(sum(gr) / len(gr), 4) if gr else None,
            "quote_grounded": round(sum(qg) / len(qg), 4) if qg else None,
            "gold_in_context": round(sum(r["gold_in_context"] for r in rows) / n, 4),
            "avg_prompt_tokens": round(sum(r["prompt_tokens"] for r in rows) / n, 1),
            "avg_out_tokens": round(sum(r["out_tokens"] for r in rows) / n, 1)}


# score every file in pred_dir and write one combined metrics.json
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--pred_dir", default="work/preds")
    ap.add_argument("--out", default="results/metrics.json")
    a = ap.parse_args()
    out = []
    for fn in sorted(Path(a.pred_dir).glob("*.jsonl")):
        rows = read_jsonl(fn)
        if rows:
            out.append({"tag": rows[0]["tag"], "split": rows[0]["split"], "run": rows[0]["run"], **score(rows)})
            print(out[-1])
    write_json(a.out, out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/proj/src/bench.py
"""Stage 8 - systems benchmark on ONE T4: p50/p95 latency, throughput, peak VRAM, cost per 1,000 queries.

Run with  CUDA_VISIBLE_DEVICES=0  so only a single T4 is visible.
  --backend vllm : latency (sequential, concurrency 1) + throughput (all queries batched). vLLM pre-allocates the
                   KV cache, so its nvidia-smi number is NOT a meaningful 'peak VRAM'.
  --backend hf   : transformers generate, batch 1. torch.cuda.max_memory_allocated() = memory the pipeline really
                   needs (weights + activations + KV cache + retriever). Use this for the VRAM column.
End-to-end latency = live retrieval (embed query + search) + prompt build + generation.

    CUDA_VISIBLE_DEVICES=0 python src/bench.py --backend vllm --model work/merged/raft_cot --tag raft_cot --style cot --k 4
    CUDA_VISIBLE_DEVICES=0 python src/bench.py --backend hf   --model work/merged/raft_cot --tag raft_cot --style cot --k 4
"""
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import torch

from common import Retriever, build_messages, load_cfg, percentile, read_jsonl


# time the full pipeline (retrieve + prompt + generate) on one T4 and append one result line
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/config.yaml")
    ap.add_argument("--data_dir", default="work/data")
    ap.add_argument("--out", default="results/bench.jsonl", help="one JSON line is appended per run")
    ap.add_argument("--backend", choices=["vllm", "hf"], required=True)
    ap.add_argument("--model", required=True)
    ap.add_argument("--tag", required=True)
    ap.add_argument("--style", default="direct", choices=["direct", "cot"])
    ap.add_argument("--k", type=int, default=4)
    ap.add_argument("--quantization", default=None)
    ap.add_argument("--eager", action="store_true")
    a = ap.parse_args()
    cfg = load_cfg(a.config)
    bc, ev, rc = cfg["bench"], cfg["eval"], cfg["retrieval"]
    dd = Path(a.data_dir)
    assert torch.cuda.device_count() == 1, "set CUDA_VISIBLE_DEVICES=0 so the benchmark sees exactly one T4"

    passages = read_jsonl(dd / "passages.jsonl")
    by_pid = {p["pid"]: p for p in passages}
    rows = read_jsonl(dd / "test_seen.jsonl")[: bc["n_queries"]]
    max_new = ev["max_new_tokens_cot" if a.style == "cot" else "max_new_tokens_direct"]

    # retriever lives on the same GPU, exactly as in a real single-GPU deployment
    ret = Retriever(passages, rc["embed_model"], rc["query_prefix"], emb_path=dd / "passage_emb.npy")
    pid_list = [p["pid"] for p in passages]

    if a.backend == "vllm":
        from vllm import LLM, SamplingParams

        # a bit less VRAM than in run_eval: the retriever shares this GPU here
        llm = LLM(model=a.model, dtype="half", max_model_len=ev["max_model_len"], gpu_memory_utilization=ev["gpu_util"] - 0.08,
                  quantization=a.quantization, enforce_eager=a.eager, seed=cfg["seed"])
        tok = llm.get_tokenizer()
        sp = SamplingParams(temperature=0.0, max_tokens=max_new)

        # vLLM generate; returns (prompt tokens, output tokens) per prompt
        def gen(prompts):
            outs = llm.generate(prompts, sp, use_tqdm=False)
            return [(len(o.prompt_token_ids), len(o.outputs[0].token_ids)) for o in outs]
    else:
        from transformers import AutoModelForCausalLM, AutoTokenizer

        tok = AutoTokenizer.from_pretrained(a.model)
        model = AutoModelForCausalLM.from_pretrained(a.model, torch_dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa").eval()

        # plain HF generate, one prompt at a time - slower, but its memory numbers are honest
        @torch.no_grad()
        def gen(prompts):
            res = []
            for p in prompts:
                ids = tok(p, return_tensors="pt").to("cuda")
                out = model.generate(**ids, max_new_tokens=max_new, do_sample=False)
                res.append((ids["input_ids"].shape[1], out.shape[1] - ids["input_ids"].shape[1]))
            return res

    # live retrieval for one query + building its prompt; returns the prompt and retrieval time in ms
    def retrieve_and_prompt(ex):
        # CUDA is async - without the syncs the timer measures kernel launches, not the search
        torch.cuda.synchronize()
        t = time.perf_counter()
        ix = ret.search(ex["retrieval_query"], a.k)
        torch.cuda.synchronize()
        t_ret = (time.perf_counter() - t) * 1000
        msgs = build_messages(ex["history"], [by_pid[pid_list[i]] for i in ix], a.style)
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True), t_ret

    # warm-up queries aren't timed (CUDA graph capture, allocator, first-call overhead)
    for ex in rows[: bc["warmup"]]:
        gen([retrieve_and_prompt(ex)[0]])
    torch.cuda.reset_peak_memory_stats()

    # ---- latency: one query at a time (concurrency 1)
    ret_ms, gen_ms, e2e_ms, p_tok, o_tok = [], [], [], [], []
    for ex in rows:
        t0 = time.perf_counter()
        prompt, t_ret = retrieve_and_prompt(ex)
        t1 = time.perf_counter()
        (pt, ot), = gen([prompt])
        torch.cuda.synchronize()
        t2 = time.perf_counter()
        ret_ms.append(t_ret); gen_ms.append((t2 - t1) * 1000); e2e_ms.append((t2 - t0) * 1000); p_tok.append(pt); o_tok.append(ot)

    rec = {"tag": a.tag, "backend": a.backend, "style": a.style, "k": a.k, "n": len(rows), "model": a.model,
           "gpu": torch.cuda.get_device_name(0),
           "p50_ms": round(percentile(e2e_ms, 50), 1), "p95_ms": round(percentile(e2e_ms, 95), 1),
           "retrieval_p50_ms": round(percentile(ret_ms, 50), 1), "gen_p50_ms": round(percentile(gen_ms, 50), 1),
           "avg_prompt_tokens": round(sum(p_tok) / len(p_tok), 1), "avg_out_tokens": round(sum(o_tok) / len(o_tok), 1),
           # vLLM's engine may live in another process and pre-allocates its KV cache -> torch counters are only
           # meaningful for the HF backend, so they are left empty for vLLM.
           "peak_vram_gb_torch": round(torch.cuda.max_memory_allocated() / 2**30, 2) if a.backend == "hf" else None,
           "peak_vram_gb_reserved": round(torch.cuda.max_memory_reserved() / 2**30, 2) if a.backend == "hf" else None}

    # ---- throughput + cost (vLLM only: continuous batching is what a server would do)
    if a.backend == "vllm":
        prompts = [retrieve_and_prompt(ex)[0] for ex in rows]
        t = time.perf_counter()
        gen(prompts)
        wall = time.perf_counter() - t
        qps = len(prompts) / wall
        rec.update({"batched_qps": round(qps, 2), "usd_per_gpu_hour_assumed": bc["gpu_usd_per_hour"],
                    "cost_usd_per_1k_queries": round(bc["gpu_usd_per_hour"] / 3600 / qps * 1000, 4)})
    print(json.dumps(rec, indent=2))
    Path(a.out).parent.mkdir(parents=True, exist_ok=True)
    with open(a.out, "a") as f:
        f.write(json.dumps(rec) + "\n")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/proj/src/judge.py
"""OPTIONAL - LLM-as-judge accuracy (0 = wrong, 1 = partly right, 2 = correct) with Qwen2.5-7B-Instruct-AWQ on one T4.
Token-F1 punishes valid paraphrases; the judge compares meaning against the human reply and the gold passage.
Run after run_eval.py, in a fresh process. Judge a subset (--limit 150) to save GPU time.

    python src/judge.py --pred_dir work/preds --out results/judge.json --limit 150
"""
from __future__ import annotations

import argparse
import re
from pathlib import Path

from common import load_cfg, read_jsonl, write_json

# kept short and strict: compare facts with the human reply, answer with one digit
PROMPT = """You grade a customer-support reply.
Reference reply (from a human agent, correct): {gold}
Candidate reply: {cand}
Score the candidate: 2 = conveys the same key facts as the reference, no contradiction; 1 = partly right or missing a key fact; 0 = wrong, contradicts, or unrelated.
Answer with a single digit."""


# load the 7B judge once, grade the first --limit answers of every predictions file, save the averages
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/config.yaml")
    ap.add_argument("--pred_dir", default="work/preds")
    ap.add_argument("--out", default="results/judge.json")
    ap.add_argument("--judge_model", default="Qwen/Qwen2.5-7B-Instruct-AWQ")
    ap.add_argument("--limit", type=int, default=150)
    a = ap.parse_args()
    cfg = load_cfg(a.config)
    from vllm import LLM, SamplingParams

    llm = LLM(model=a.judge_model, quantization="awq", dtype="half", max_model_len=2048, gpu_memory_utilization=0.88, seed=0)
    tok = llm.get_tokenizer()
    sp = SamplingParams(temperature=0.0, max_tokens=2)  # a single digit is all we want back
    res = {}
    for fn in sorted(Path(a.pred_dir).glob("*.jsonl")):
        rows = read_jsonl(fn)[: a.limit]
        prompts = [tok.apply_chat_template([{"role": "user", "content": PROMPT.format(gold=r["gold_response"], cand=r["answer"] or "(empty)")}],
                                           tokenize=False, add_generation_prompt=True) for r in rows]
        scores = []
        for o in llm.generate(prompts, sp, use_tqdm=False):
            m = re.search(r"[012]", o.outputs[0].text)
            # unparseable counts as 0: harsh, but it can never inflate a score
            scores.append(int(m.group()) if m else 0)
        r0 = rows[0]
        res[f"{r0['tag']}__{r0['split']}__{r0['run']}"] = round(sum(scores) / len(scores), 3)
        print(fn.name, res[f"{r0['tag']}__{r0['split']}__{r0['run']}"])
    write_json(a.out, res)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/proj/src/report.py
"""Stage 9 - merge metrics + benchmark (+ optional judge) into results/RESULTS.md and two charts.

    python src/report.py --metrics results/metrics.json --bench results/bench.jsonl --judge results/judge.json
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

# one fixed colour + marker per model, so a model looks the same in both charts
SLOTS = {"base": "#2a78d6", "raft_cot": "#eb6834", "qwen3b": "#e87ba4", "qwen7b_awq": "#1baf7a", "raft_direct": "#eda100"}
MARK = {"base": "o", "raft_cot": "s", "qwen3b": "v", "qwen7b_awq": "^", "raft_direct": "D"}
INK, MUTED, SURFACE = "#0b0b0b", "#52514e", "#fcfcfb"


# shared look for both charts: light background, no top/right border, faint horizontal grid
def style(ax):
    ax.set_facecolor(SURFACE)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color("#c9c8c2")
    ax.tick_params(colors=MUTED)
    ax.grid(axis="y", color="#e6e5e0", linewidth=0.8)
    ax.set_axisbelow(True)


# read metrics + bench (+ judge) -> write RESULTS.md with two tables -> draw two charts
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--metrics", default="results/metrics.json")
    ap.add_argument("--bench", default="results/bench.jsonl")
    ap.add_argument("--judge", default="results/judge.json")
    ap.add_argument("--out_dir", default="results")
    a = ap.parse_args()
    out = Path(a.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    M = json.load(open(a.metrics))
    B = [json.loads(l) for l in open(a.bench)] if Path(a.bench).exists() else []
    J = json.load(open(a.judge)) if Path(a.judge).exists() else {}
    # judge scores are stored under 'tag__split__run'
    jkey = lambda m: f"{m['tag']}__{m['split']}__{m['run']}"

    L = ["# RAFT vs RAG on a single T4 - results\n", "## Answer quality (Doc2Dial, held-out dialogues)\n",
         "| model | split | context | n | F1 [95% CI] | ROUGE-L | judge (0-2) | grounded | quote ok | gold in ctx | out tok |", "|---|---|---|---|---|---|---|---|---|---|---|"]
    # print a number with 3 decimals, or '-' when that metric doesn't apply
    f = lambda v: "-" if v is None else f"{v:.3f}"
    for m in sorted(M, key=lambda m: (m["split"], m["run"], m["tag"])):
        L.append(f"| {m['tag']} | {m['split']} | {m['run']} | {m['n']} | {m['f1']:.3f} [{m['f1_lo']:.3f}, {m['f1_hi']:.3f}] | {m['rougeL']:.3f} | "
                 f"{f(J.get(jkey(m)))} | {f(m['groundedness'])} | {f(m['quote_grounded'])} | {m['gold_in_context']:.2f} | {m['avg_out_tokens']:.0f} |")
    L += ["\n## Systems cost (one T4, concurrency 1 for latency)\n",
          "| model | backend | k | p50 ms | p95 ms | retrieval p50 ms | prompt tok | out tok | peak VRAM GB (HF) | batched q/s | $ / 1k queries |", "|---|---|---|---|---|---|---|---|---|---|---|"]
    for b in B:
        L.append(f"| {b['tag']} | {b['backend']} | {b['k']} | {b['p50_ms']} | {b['p95_ms']} | {b['retrieval_p50_ms']} | {b['avg_prompt_tokens']} | {b['avg_out_tokens']} | "
                 f"{b.get('peak_vram_gb_torch') or '-'} | {b.get('batched_qps', '-')} | {b.get('cost_usd_per_1k_queries', '-')} |")
    if B:
        L.append(f"\nCost assumes ${B[0].get('usd_per_gpu_hour_assumed', 'n/a')}/GPU-hour (edit configs/config.yaml). GPU: {B[0]['gpu']}.")
    (out / "RESULTS.md").write_text("\n".join(L) + "\n")

    # chart 1: quality vs latency (one point per model, k=4 RAG-style context, seen docs)
    vb = {b["tag"]: b for b in B if b["backend"] == "vllm"}
    pts = {m["tag"]: m for m in M if m["split"] == "test_seen" and m["run"] == "rag:4"}
    fig, ax = plt.subplots(figsize=(7, 4.2), facecolor=SURFACE)
    style(ax)
    for tag, m in pts.items():
        if tag in vb:
            x, y, c = vb[tag]["p50_ms"], m["f1"], SLOTS.get(tag, "#6250d6")
            ax.errorbar(x, y, yerr=[[y - m["f1_lo"]], [m["f1_hi"] - y]], color=c, capsize=3, lw=1.5)
            ax.scatter(x, y, s=70, color=c, marker=MARK.get(tag, "o"), edgecolor=SURFACE, linewidth=2, label=tag, zorder=3)
            ax.annotate(tag, (x, y), xytext=(8, 6), textcoords="offset points", color=INK, fontsize=9)
    ax.set_xlabel("p50 end-to-end latency (ms)", color=MUTED)
    ax.set_ylabel("token F1 vs human agent reply", color=MUTED)
    ax.set_title("Quality vs latency on one T4 (top-4 retrieval, seen docs)", color=INK, loc="left", fontsize=11)
    if len(pts) > 1:
        ax.legend(frameon=False, labelcolor=MUTED)
    fig.tight_layout()
    fig.savefig(out / "quality_vs_latency.png", dpi=160)

    # chart 2: F1 by context mode, per split
    runs = ["closed", "rag:2", "rag:4", "rag:8", "oracle"]
    tags = [t for t in SLOTS if any(m["tag"] == t for m in M)] + sorted({m["tag"] for m in M} - set(SLOTS))
    fig, axs = plt.subplots(1, 2, figsize=(10, 4), facecolor=SURFACE, sharey=True)
    for ax, split in zip(axs, ["test_seen", "test_unseen"]):
        style(ax)
        w = 0.8 / max(1, len(tags))
        for ti, t in enumerate(tags):
            vals = [next((m["f1"] for m in M if m["tag"] == t and m["split"] == split and m["run"] == r), 0) for r in runs]
            ax.bar([i + ti * w for i in range(len(runs))], vals, w * 0.92, color=SLOTS.get(t, "#6250d6"), label=t)
        ax.set_xticks([i + w * (len(tags) - 1) / 2 for i in range(len(runs))], runs)
        ax.set_title(split.replace("_", " "), color=INK, loc="left", fontsize=11)
    axs[0].set_ylabel("token F1", color=MUTED)
    axs[0].legend(frameon=False, labelcolor=MUTED)
    fig.tight_layout()
    fig.savefig(out / "f1_by_context.png", dpi=160)
    print((out / "RESULTS.md").read_text())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/proj/src/prepare_data.py
"""Stage 1 - download Doc2Dial and turn it into: passages (the RAG corpus) + QA examples.

Each example = one agent turn that is grounded in the documents:
  history (previous turns) -> gold agent reply, gold passage ids, gold quote (the grounding spans).

Splits (all written to --out_dir):
  train / valid      dialogues from doc2dial train on *seen* docs
  test_seen          doc2dial validation dialogues on seen docs   (same corpus RAFT trained on)
  test_unseen        dialogues on held-out docs never used in training (tests generalisation)

    python src/prepare_data.py --out_dir work/data
    python src/prepare_data.py --raw_dir /kaggle/input/doc2dial-raw --out_dir work/data   # offline fallback
"""
from __future__ import annotations

import argparse
import random
import urllib.request
import zipfile
from pathlib import Path

from common import load_cfg, read_jsonl, retrieval_query, set_seed, write_json, write_jsonl  # noqa: F401
import json


# download the Doc2Dial zip once (skipped if already there) and unpack it
def download_and_extract(url: str, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    zpath = dest / "doc2dial.zip"
    if not zpath.exists():
        req = urllib.request.Request(url, headers={"User-Agent": "raft-vs-rag/1.0"})
        with urllib.request.urlopen(req, timeout=120) as r, open(zpath, "wb") as f:
            f.write(r.read())
    root = dest.resolve()
    with zipfile.ZipFile(zpath) as zf:
        for m in zf.namelist():  # zip-slip check before extracting
            if not (root / m).resolve().is_relative_to(root):
                raise RuntimeError(f"unsafe path in zip: {m}")
        zf.extractall(dest)


# find a file anywhere under root - the zip's folder layout isn't something to rely on
def find_one(root: Path, name: str) -> Path:
    hits = sorted(root.rglob(name))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {root}")
    return hits[0]


# the json files wrap everything in one top-level key ('doc_data' / 'dial_data'); strip it if present
def unwrap(obj: dict, key: str) -> dict:
    return obj.get(key, obj)


# int() that doesn't crash on missing or odd values - only used for sorting spans
def _num(x, default=0) -> int:
    try:
        return int(x)
    except (TypeError, ValueError):
        return default


# titles/ids can carry a '#...' suffix (e.g. 'Some Title#3_0'); keep just the readable part
def clean_title(t: str) -> str:
    return (t or "").split("#")[0].strip()


# turn every document into ~max_words passages that never cross a section boundary
def build_passages(doc_data: dict, max_words: int) -> tuple[list[dict], dict]:
    """Chunk each document by section. Returns (passages, {(doc_id, sp_id): passage_id})."""
    # doc2dial spans are sentence-sized, too small to retrieve on their own, so they get stitched back
    # together within a section up to ~max_words. sp2pid remembers which passage each span ended up in,
    # which is how the gold labels survive the chunking.
    passages, sp2pid = [], {}
    for domain, docs in doc_data.items():
        for doc_id, doc in docs.items():
            spans = doc.get("spans") or {}
            items = sorted(spans.items(), key=lambda kv: _num(kv[1].get("start_sp")))
            cur, n = None, 0

            # close the passage being built and start fresh
            def flush():
                nonlocal cur, n
                if cur and cur["words"] > 0:
                    cur["text"] = " ".join(cur.pop("_t"))
                    passages.append(cur)
                cur = None

            for sp_key, sp in items:
                text = " ".join((sp.get("text_sp") or "").split())
                if not text:
                    continue
                sec = str(sp.get("id_sec", ""))
                if cur is None or sec != cur["_sec"] or cur["words"] >= max_words:
                    flush()
                    pid = f"{doc_id}::{n}"
                    n += 1
                    cur = {"pid": pid, "doc_id": doc_id, "domain": domain, "title": clean_title(doc.get("title", doc_id)),
                           "section": " ".join((sp.get("title") or "").split()), "_sec": sec, "words": 0, "_t": []}
                cur["_t"].append(text)
                cur["words"] += len(text.split())
                sp2pid[(doc_id, str(sp_key))] = cur["pid"]
            flush()
    for p in passages:
        p.pop("_sec", None)
    return passages, sp2pid


# one example per grounded agent turn: recent history -> human reply + which passages it came from
def build_examples(dial_data: dict, doc_data: dict, sp2pid: dict, hist_turns: int, source: str) -> list[dict]:
    span_text = {(d, str(k)): " ".join((s.get("text_sp") or "").split())
                 for dom in doc_data.values() for d, doc in dom.items() for k, s in (doc.get("spans") or {}).items()}
    out = []
    for domain, docs in dial_data.items():
        for doc_id, dials in docs.items():
            for dial in dials:
                turns = dial.get("turns", [])
                did = dial.get("dial_id", "")
                for i, t in enumerate(turns):
                    # only agent turns that answer a customer turn
                    if t.get("role") != "agent" or i == 0 or turns[i - 1].get("role") != "user":
                        continue
                    resp = " ".join((t.get("utterance") or "").split())
                    # skip "ok" / "thanks" style replies and the odd very long one
                    if not (3 <= len(resp.split()) <= 120):
                        continue
                    refs = [str(r.get("sp_id")) for r in (t.get("references") or []) if r.get("sp_id") is not None]
                    gold = list(dict.fromkeys(sp2pid[(doc_id, r)] for r in refs if (doc_id, r) in sp2pid))
                    # no grounding span -> nothing to score retrieval or grounding against
                    if not gold:
                        continue
                    quote = " ".join(dict.fromkeys(span_text.get((doc_id, r), "") for r in refs if span_text.get((doc_id, r))))
                    quote = " ".join(quote.split()[:70])
                    hist = [{"role": x["role"], "text": " ".join((x.get("utterance") or "").split())}
                            for x in turns[max(0, i - hist_turns) : i]]
                    out.append({"id": f"{did}_{t.get('turn_id', i)}", "dial_id": did, "doc_id": doc_id, "domain": domain,
                                "history": hist, "response": resp, "gold_pids": gold, "gold_quote": quote,
                                "retrieval_query": retrieval_query(hist), "source": source})
    return out


# download -> chunk -> build examples -> split (holding out whole documents) -> write files + stats
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/config.yaml")
    ap.add_argument("--out_dir", default="work/data")
    ap.add_argument("--raw_dir", default=None, help="folder that already contains doc2dial_*.json (skips download)")
    a = ap.parse_args()
    cfg = load_cfg(a.config)
    d = cfg["data"]
    set_seed(cfg["seed"])
    rng = random.Random(cfg["seed"])
    out = Path(a.out_dir)

    raw = Path(a.raw_dir) if a.raw_dir else out.parent / "raw"
    if not a.raw_dir:
        download_and_extract(d["doc2dial_url"], raw)
    # raw doc2dial: the documents, plus train and validation dialogues
    doc_data = unwrap(json.load(open(find_one(raw, "doc2dial_doc.json"))), "doc_data")
    tr_dial = unwrap(json.load(open(find_one(raw, "doc2dial_dial_train.json"))), "dial_data")
    va_dial = unwrap(json.load(open(find_one(raw, "doc2dial_dial_validation.json"))), "dial_data")

    passages, sp2pid = build_passages(doc_data, d["passage_words"])
    all_docs = sorted({p["doc_id"] for p in passages})
    # held out by document, not by dialogue - otherwise "unseen" would still share docs with train
    holdout = set(rng.sample(all_docs, max(1, int(len(all_docs) * d["holdout_doc_frac"]))))
    write_jsonl(out / "passages.jsonl", passages)

    # turn dialogues into examples, then sort them into pools by whether their doc is held out
    tr = build_examples(tr_dial, doc_data, sp2pid, d["history_turns"], "train")
    va = build_examples(va_dial, doc_data, sp2pid, d["history_turns"], "validation")

    unseen_pool = [e for e in tr + va if e["doc_id"] in holdout]
    train_pool = [e for e in tr if e["doc_id"] not in holdout]
    seen_pool = [e for e in va if e["doc_id"] not in holdout]

    # valid is split by dialogue so turns of one conversation don't land on both sides
    dials = sorted({e["dial_id"] for e in train_pool})
    valid_d = set(rng.sample(dials, max(1, int(len(dials) * d["valid_dialogue_frac"]))))
    valid_pool = [e for e in train_pool if e["dial_id"] in valid_d]
    train_pool = [e for e in train_pool if e["dial_id"] not in valid_d]

    # random subset of at most n rows, so split sizes stay under the config limits
    def cap(rows, n):
        rows = rows[:]
        rng.shuffle(rows)
        return rows[:n]

    splits = {"train": cap(train_pool, d["max_train"]), "valid": cap(valid_pool, d["max_valid"]),
              "test_seen": cap(seen_pool, d["max_test_seen"]), "test_unseen": cap(unseen_pool, d["max_test_unseen"])}
    for name, rows in splits.items():
        write_jsonl(out / f"{name}.jsonl", rows)

    stats = {"n_docs": len(all_docs), "n_holdout_docs": len(holdout), "n_passages": len(passages),
             "avg_passage_words": round(sum(p["words"] for p in passages) / max(1, len(passages)), 1),
             **{f"n_{k}": len(v) for k, v in splits.items()},
             "pool_sizes": {"train": len(train_pool), "seen": len(seen_pool), "unseen": len(unseen_pool)},
             "domains": {dom: sum(p["domain"] == dom for p in passages) for dom in {p["domain"] for p in passages}}}
    write_json(out / "stats.json", stats)
    print(json.dumps(stats, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/proj/src/build_index.py
"""Stage 2 - embed the corpus, retrieve top-N passages for every query, report retrieval quality.

Writes  ret_<split>.jsonl  ({id, retrieved: [pid...], scores}) and retrieval_report.json
The same retrieval results feed (a) hard negatives for RAFT training and (b) the RAG baseline.

    python src/build_index.py --data_dir work/data
"""
from __future__ import annotations

import argparse
from pathlib import Path

from common import Retriever, load_cfg, read_jsonl, write_json, write_jsonl

# every split gets retrieved: train/valid for RAFT distractors, the test splits for RAG eval
SPLITS = ["train", "valid", "test_seen", "test_unseen"]


# embed the corpus, retrieve top-N for every example, and measure how often the gold passage shows up
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/config.yaml")
    ap.add_argument("--data_dir", default="work/data")
    a = ap.parse_args()
    cfg = load_cfg(a.config)
    rc = cfg["retrieval"]
    dd = Path(a.data_dir)

    passages = read_jsonl(dd / "passages.jsonl")
    pids = [p["pid"] for p in passages]
    pid_doc = {p["pid"]: p["doc_id"] for p in passages}
    ret = Retriever(passages, rc["embed_model"], rc["query_prefix"], emb_path=dd / "passage_emb.npy")

    report = {}
    for split in SPLITS:
        rows = read_jsonl(dd / f"{split}.jsonl")
        idxs, scores = ret.search_batch([r["retrieval_query"] for r in rows], rc["save_top"])
        # counters for recall@k (gold found in top k), MRR (how high the first gold hit ranks), same-doc rate
        out, hits, mrr, dochit = [], {1: 0, 2: 0, 4: 0, 8: 0, 20: 0}, 0.0, 0
        for r, ix, sc in zip(rows, idxs, scores):
            got = [pids[i] for i in ix]
            out.append({"id": r["id"], "retrieved": got, "scores": [round(s, 4) for s in sc]})
            gold = set(r["gold_pids"])
            for k in hits:
                hits[k] += bool(gold & set(got[:k]))
            rank = next((j + 1 for j, p in enumerate(got) if p in gold), None)
            mrr += 1 / rank if rank else 0
            # even when the exact passage is missed, is the top hit at least from the right document?
            dochit += pid_doc[got[0]] == r["doc_id"]
        write_jsonl(dd / f"ret_{split}.jsonl", out)
        n = max(1, len(rows))
        report[split] = {"n": len(rows), **{f"recall@{k}": round(v / n, 4) for k, v in hits.items()},
                         "mrr": round(mrr / n, 4), "top1_same_doc": round(dochit / n, 4)}
        print(split, report[split])
    write_json(dd / "retrieval_report.json", report)


if __name__ == "__main__":
    main()

### Data + model paths
`RAFT` points at the HF repo the merged model from notebook 2 was pushed to.

In [ ]:
import glob
# first file under /kaggle/input that matches the pattern; fails loudly if the input isn't attached
def find(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    assert hits, f"nothing matches {pattern} - did you add the previous notebook as an Input?"
    return hits[0]

import glob, subprocess
hits = sorted(glob.glob('/kaggle/input/**/passages.jsonl', recursive=True))
if hits:
    DATA = os.path.dirname(hits[0]); print('using notebook-1 data from', DATA)
else:
    # notebook-1 output isn't mounted -> rebuild it here. same seed, so the splits come out identical (~5 min, needs internet)
    print('passages.jsonl not found under /kaggle/input; mounted items:'); print(*sorted(glob.glob('/kaggle/input/*/*'))[:20], sep='\n')
    DATA = '/kaggle/working/proj/work/data'
    subprocess.run(['python', 'src/prepare_data.py', '--out_dir', DATA], check=True)
    subprocess.run(['python', 'src/build_index.py', '--data_dir', DATA], check=True)
RAFT = "your-real-username/raft-qwen2.5-1.5b-doc2dial"                    # <- HF repo the merged model was pushed to

# private repo only: needs the HF_TOKEN secret attached in Kaggle
# from kaggle_secrets import UserSecretsClient
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

assert '<' not in RAFT and 'your-real-username' not in RAFT, "edit RAFT above to your real HF repo name"
print('data :', DATA, '\nmodel:', RAFT)

### 3.0 Smoke test (optional, ~2 min)
20 queries, just to confirm vLLM runs on this T4 before the long cells. If it OOMs, add `--eager`.

In [ ]:
!python src/run_eval.py --data_dir {DATA} --model Qwen/Qwen2.5-1.5B-Instruct --tag smoke --style direct --runs rag:4 --splits test_seen --limit 20 --out_dir work/smoke

### 3.1 Base 1.5B + RAG
Top-2/4/8 retrieval, plus closed-book (no docs, the floor) and oracle (gold passage only, the ceiling).

In [ ]:
!python src/run_eval.py --data_dir {DATA} --model Qwen/Qwen2.5-1.5B-Instruct --tag base --style direct --runs rag:4 closed oracle rag:2 rag:8

### 3.2 RAFT 1.5B - same retrieved passages as the base model

In [ ]:
!python src/run_eval.py --data_dir {DATA} --model {RAFT} --tag raft_cot --style cot --runs rag:4 closed oracle rag:2 rag:8

### 3.3 Qwen2.5-3B + RAG
Middle of the size ladder. Fits on the T4 in fp16, no quantisation needed.

In [ ]:
!python src/run_eval.py --data_dir {DATA} --model Qwen/Qwen2.5-3B-Instruct --tag qwen3b --style direct --runs rag:4

### 3.4 Qwen2.5-7B (4-bit AWQ) + RAG
The comparison that actually matters for Uniphore: does a fine-tuned 1.5B hold up against a bigger general model with plain RAG, at a fraction of the cost?

In [ ]:
!python src/run_eval.py --data_dir {DATA} --model Qwen/Qwen2.5-7B-Instruct-AWQ --quantization awq --tag qwen7b_awq --style direct --runs rag:4

### 3.5 Score everything
Token-F1 with a bootstrap CI, ROUGE-L, groundedness, and the quote-grounded rate for the RAFT model.

In [ ]:
!python src/metrics.py --pred_dir work/preds --out results/metrics.json

### 3.6 Latency / throughput / cost (vLLM, one T4)
Every run spins up its own engine, so this is the slow cell.

In [ ]:
!python src/bench.py --backend vllm --data_dir {DATA} --model Qwen/Qwen2.5-1.5B-Instruct --tag base --style direct --k 4
!python src/bench.py --backend vllm --data_dir {DATA} --model {RAFT} --tag raft_cot --style cot --k 4
!python src/bench.py --backend vllm --data_dir {DATA} --model Qwen/Qwen2.5-3B-Instruct --tag qwen3b --style direct --k 4
!python src/bench.py --backend vllm --data_dir {DATA} --model Qwen/Qwen2.5-7B-Instruct-AWQ --quantization awq --tag qwen7b_awq --style direct --k 4

### 3.7 Peak VRAM (HF backend)
vLLM pre-allocates its KV cache, so its memory number says nothing about what the model needs; plain HF `generate` does. The 7B AWQ run needs `autoawq` on this path and failed last time, so it's commented out - the 1.5B vs 3B numbers are the ones that matter here anyway.

In [ ]:
!python src/bench.py --backend hf --data_dir {DATA} --model Qwen/Qwen2.5-1.5B-Instruct --tag base --style direct --k 4
!python src/bench.py --backend hf --data_dir {DATA} --model {RAFT} --tag raft_cot --style cot --k 4
!python src/bench.py --backend hf --data_dir {DATA} --model Qwen/Qwen2.5-3B-Instruct --tag qwen3b --style direct --k 4
# !python src/bench.py --backend hf --data_dir {DATA} --model Qwen/Qwen2.5-7B-Instruct-AWQ --tag qwen7b_awq --style direct --k 4   # needs autoawq

### 3.8 LLM judge (0-2) on the first 150 queries of every run
Token-F1 punishes valid paraphrases, so Qwen2.5-7B-AWQ grades meaning against the human reply. All files are re-judged together so every score comes from the same judge.

In [ ]:
!python src/judge.py --pred_dir work/preds --out results/judge.json --limit 150

### 3.9 Report - tables + charts

How I read the table:
- **F1 / ROUGE-L** - word overlap with the human agent's reply. Low-ish absolute numbers are normal here (human replies are free-form); the gaps between models are what count, and the CI shows whether a gap is real.
- **judge** - 0-2 meaning-level grade from the 7B model. Catches correct paraphrases that F1 misses.
- **grounded** - share of answer words found in the given passages. A drop usually means the model is answering from memory.
- **quote ok** - RAFT only: did its quotes really come from the context?
- **gold in ctx** - how often retrieval actually delivered the right passage. Same for every model with the same k, which is the point.
- **closed / oracle** - floor and ceiling. Where `rag:4` sits between them shows how much is lost to retrieval vs to the model.

In [ ]:
!python src/report.py --metrics results/metrics.json --bench results/bench.jsonl --judge results/judge.json

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open('results/RESULTS.md').read()))
display(Image('results/quality_vs_latency.png')); display(Image('results/f1_by_context.png'))
import json; print(json.dumps(json.load(open(DATA + '/retrieval_report.json')), indent=1))

### 3.10 Read the actual outputs
8 examples side by side: question, gold reply, base, RAFT. The metrics can hide things (e.g. a model copying the quote and calling it an answer), so these get read before trusting the numbers.

In [ ]:
import json
# quick jsonl reader for this cell
rd = lambda f: [json.loads(l) for l in open(f)]
a, b = rd('work/preds/base__test_seen__rag4.jsonl'), rd('work/preds/raft_cot__test_seen__rag4.jsonl')
# RAFT shows 'raw' so the quote is visible too, not just the parsed answer
for x, y in list(zip(a, b))[:8]:
    print('Q    :', x['history'][-1]['text']); print('GOLD :', x['gold_response']); print('BASE :', x['answer']); print('RAFT :', y['raw'])
    print('gold passage in context:', x['gold_in_context'], '\n' + '-' * 90)

Everything under `results/` is kept with Save Version. `RESULTS.md` and the two PNGs are in the Output tab.